In [0]:
%run ./Classroom-Setup-5

In [0]:
-- Lightweight helper for the cross-system join in 5.5 Section D.
-- The lab joins ~50 customer_enrichment rows in Lakebase against the
-- TPC-DS store_sales fact, scoped to a 30-day window. Rather than scan
-- the full ~2.8B-row store_sales_unclustered, build (or reuse) a narrow
-- pre-filtered slice with just the columns the join needs.
--
-- Idempotent: rebuilds only if the table is missing or has the wrong
-- row count. The expected count is fixed by the date filter, so it is
-- safe to compare against the live source.
BEGIN
  DECLARE table_exists      BOOLEAN DEFAULT FALSE;
  DECLARE actual_rows       BIGINT  DEFAULT -1;
  DECLARE expected_rows     BIGINT;
  DECLARE needs_rebuild     BOOLEAN DEFAULT TRUE;

  SET expected_rows = (
    SELECT COUNT(*)
    FROM samples.tpcds_sf1000.store_sales
    WHERE ss_sold_date_sk BETWEEN 2451180 AND 2451210
  );

  SET table_exists = (
    SELECT COUNT(*) > 0
    FROM system.information_schema.tables
    WHERE table_catalog = current_catalog()
      AND table_schema  = 'data_interoperability_tpcds'
      AND table_name    = 'lab_store_sales_slice'
  );

  IF table_exists THEN
    SET actual_rows = (
      SELECT COUNT(*) FROM data_interoperability_tpcds.lab_store_sales_slice
    );
    IF actual_rows = expected_rows THEN
      SET needs_rebuild = FALSE;
    END IF;
  END IF;

  IF needs_rebuild THEN
    EXECUTE IMMEDIATE
      'CREATE OR REPLACE TABLE data_interoperability_tpcds.lab_store_sales_slice AS '
      || 'SELECT ss_customer_sk, ss_sold_date_sk, ss_net_paid '
      || 'FROM samples.tpcds_sf1000.store_sales '
      || 'WHERE ss_sold_date_sk BETWEEN 2451180 AND 2451210';
  END IF;

  SELECT
    'data_interoperability_tpcds.lab_store_sales_slice' AS table_name,
    expected_rows                                       AS expected_rows,
    (SELECT COUNT(*) FROM data_interoperability_tpcds.lab_store_sales_slice) AS actual_rows,
    CASE WHEN needs_rebuild THEN 'rebuilt' ELSE 'reused' END AS status;
END;

In [0]:
DECLARE OR REPLACE lakebase_project_name STRING;
DECLARE OR REPLACE lab_pg_connection STRING;
DECLARE OR REPLACE lab_pg_catalog STRING;
DECLARE OR REPLACE pg_create_connection_sql STRING;
DECLARE OR REPLACE q STRING DEFAULT CHR(39);
DECLARE OR REPLACE pg_create_catalog_sql STRING;
DECLARE OR REPLACE pg_host_value STRING;
DECLARE OR REPLACE pg_oauth_token_value STRING;

SET VAR lakebase_project_name = CONCAT(
  'interop_course_',
  CASE
    WHEN current_catalog() LIKE 'labuser_%'
      THEN SUBSTRING(current_catalog(), LENGTH('labuser_') + 1)
    ELSE
      REGEXP_REPLACE(
        LOWER(SPLIT(current_user(), '@')[0]),
        '[^a-z0-9_]', '_'
      )
  END
);

SET VAR lab_pg_connection = CONCAT(
 SPLIT(current_user(), '@')[0],
 '_pg_connection'
);

SET VAR lab_pg_catalog = CONCAT(
 SPLIT(current_user(), '@')[0],
 '_pg_catalog'
);


SELECT
  current_catalog()       AS user_catalog,
  current_user()          AS user_email,
  lakebase_project_name   AS lakebase_project_name,
  lab_pg_connection       AS lab_pg_connection,
  lab_pg_catalog          AS lab_pg_catalog;
